# Tunda Box – Junior Data Analyst Assessment (solutions notebook)
All data is fictional. Amounts in KES. SQL runs on SQLite (in-memory) via Python; Postgres equivalents are noted in comments.

**Contents:** Setup & cleaning rules → S1–S5 (SQL) → D1 audit → D2/D3 → T1/T2 stats → Part 5 business case.

In [1]:
import sqlite3, pandas as pd, numpy as np
from scipy.stats import norm
pd.set_option('display.width', 140)

customers = pd.read_csv('data/customers.csv')
orders = pd.read_csv('data/orders.csv')

con = sqlite3.connect(':memory:')
customers.to_sql('customers', con, index=False)
orders.to_sql('orders', con, index=False)
q = lambda sql: pd.read_sql(sql, con)

## 0. Data audit first (this drives every later decision)
Checks run before any business number is produced.

In [2]:
print('Duplicate order_ids:'); display(q("SELECT order_id, COUNT(*) n FROM orders GROUP BY order_id HAVING n>1"))
print('Orders with no matching customer (orphans):'); display(q("SELECT o.* FROM orders o LEFT JOIN customers c USING(customer_id) WHERE c.customer_id IS NULL"))
print('NULL amounts:'); display(q("SELECT * FROM orders WHERE amount_kes IS NULL"))
print('City spellings (raw, with length to expose spaces):'); display(q("SELECT city, LENGTH(city) len, COUNT(*) n FROM customers GROUP BY city"))
print('Amount distribution (completed):'); display(orders.query("status=='completed'").amount_kes.describe())
print('Orders dated BEFORE customer signup:'); display(q("SELECT o.order_id,o.customer_id,o.order_date,c.signup_date FROM orders o JOIN customers c USING(customer_id) WHERE o.order_date < c.signup_date"))
print('Date range of orders:'); display(q("SELECT MIN(order_date) first_order, MAX(order_date) last_order FROM orders"))

Duplicate order_ids:


,order_id,n
0,1006,2


Orders with no matching customer (orphans):


,order_id,customer_id,order_date,status,amount_kes,payment_method
0,1020,999,2026-09-26,completed,1900.0,M-Pesa


NULL amounts:


,order_id,customer_id,order_date,status,amount_kes,payment_method
0,1009,107,2026-07-21,completed,None,M-Pesa


... (4 outputs left out)

Orders dated BEFORE customer signup:


,order_id,customer_id,order_date,signup_date
0,1003,103,2026-06-11,2026-06-20


Date range of orders:


,first_order,last_order
0,2026-06-03,2026-09-26


### Cleaning rules (stated assumptions)
1. **Cities** – `TRIM` + normalise case (Postgres: `INITCAP(TRIM(city))`).
2. **Duplicate rows** – keep one row per `order_id`.
3. **Revenue** = `status='completed'` only (cancelled = never fulfilled; refunded = money returned).
4. **NULL amount** (order 1009) – kept as an order but contributes no revenue; flagged, not guessed.
5. **Outlier** (order 1013, KES 185,000, ~80x the typical order) – treated as a probable entry error (likely 1,850 with extra zeros) and **excluded from headline figures, shown in sensitivity**. Rule used in SQL: `amount_kes < 50000`.
6. **Orphan order** (1020, customer 999) – excluded wherever a city/customer is needed; kept for order-only metrics like monthly revenue, and flagged.

In [3]:
con.executescript('''
CREATE VIEW customers_clean AS
SELECT customer_id,
       UPPER(SUBSTR(TRIM(city),1,1)) || LOWER(SUBSTR(TRIM(city),2)) AS city,   -- Postgres: INITCAP(TRIM(city))
       signup_date, channel
FROM customers;

CREATE VIEW orders_dedup AS
SELECT order_id, customer_id, order_date, status, amount_kes, payment_method
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_date) rn FROM orders)
WHERE rn = 1;
''')

## S1. Orders by status

In [4]:
display(q('''
SELECT status,
       COUNT(*)                 AS rows_raw,
       COUNT(DISTINCT order_id) AS orders_distinct
FROM orders
GROUP BY status
ORDER BY orders_distinct DESC;
'''))

,status,rows_raw,orders_distinct
0,completed,17,16
1,refunded,2,2
2,cancelled,2,2


## S2(a). Completed-order revenue by city

In [5]:
print('NAIVE (no cleaning) - what a rushed query returns:')
display(q('''
SELECT c.city, SUM(o.amount_kes) AS revenue
FROM orders o JOIN customers c USING(customer_id)
WHERE o.status='completed' GROUP BY c.city ORDER BY revenue DESC;'''))

print('CLEAN (headline):')
display(q('''
SELECT c.city,
       SUM(o.amount_kes) AS revenue_kes,
       COUNT(*)          AS completed_orders
FROM orders_dedup o
JOIN customers_clean c USING(customer_id)
WHERE o.status = 'completed'
  AND o.amount_kes < 50000          -- exclude suspected entry error (order 1013)
GROUP BY c.city
ORDER BY revenue_kes DESC;'''))

print('SENSITIVITY: if the 185,000 order is genuine:')
display(q('''
SELECT c.city, SUM(o.amount_kes) AS revenue_kes
FROM orders_dedup o JOIN customers_clean c USING(customer_id)
WHERE o.status='completed'
GROUP BY c.city ORDER BY revenue_kes DESC;'''))
print('SENSITIVITY: if 1013 was meant to be 1,850:')
alt = q('''SELECT c.city, SUM(CASE WHEN o.order_id=1013 THEN 1850 ELSE o.amount_kes END) AS revenue_kes
FROM orders_dedup o JOIN customers_clean c USING(customer_id)
WHERE o.status='completed' GROUP BY c.city ORDER BY revenue_kes DESC;''')
display(alt)

NAIVE (no cleaning) - what a rushed query returns:


,city,revenue
0,Kisumu,187100.0
1,Nairobi,11750.0
2,nairobi,7400.0
3,Mombasa,5150.0
4,Nakuru,3300.0
5,Nairobi,3000.0


CLEAN (headline):


,city,revenue_kes,completed_orders
0,Nairobi,20650.0,9
1,Mombasa,5150.0,2
2,Nakuru,3300.0,1
3,Kisumu,2100.0,1


SENSITIVITY: if the 185,000 order is genuine:


,city,revenue_kes
0,Kisumu,187100.0
1,Nairobi,20650.0
2,Mombasa,5150.0
3,Nakuru,3300.0


SENSITIVITY: if 1013 was meant to be 1,850:


,city,revenue_kes
0,Nairobi,20650.0
1,Mombasa,5150.0
2,Kisumu,3950.0
3,Nakuru,3300.0


## S3. Customers who never ordered

In [6]:
display(q('''
SELECT c.customer_id, c.city, c.signup_date, c.channel
FROM customers_clean c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL;'''))

,customer_id,city,signup_date,channel
0,109,Eldoret,2026-06-21,Organic
1,110,Kisumu,2026-07-05,Paid Social


## S4. Repeat purchase rate

In [7]:
display(q('''
WITH per_customer AS (
  SELECT c.customer_id, COUNT(o.order_id) AS completed_orders
  FROM customers c
  LEFT JOIN orders_dedup o ON o.customer_id = c.customer_id AND o.status='completed'
  GROUP BY c.customer_id)
SELECT COUNT(*)                                          AS customers,
       SUM(completed_orders >= 1)                        AS buyers,
       SUM(completed_orders >= 2)                        AS repeat_buyers,
       ROUND(100.0*SUM(completed_orders>=2)/COUNT(*),1)  AS repeat_pct_of_all_customers,
       ROUND(100.0*SUM(completed_orders>=2)/SUM(completed_orders>=1),1) AS repeat_pct_of_buyers
FROM per_customer;'''))
print('Sensitivity - WITHOUT deduping order 1006 (customer 105 wrongly looks like a repeat buyer):')
display(q('''
WITH p AS (SELECT c.customer_id, COUNT(o.order_id) n FROM customers c
           LEFT JOIN orders o ON o.customer_id=c.customer_id AND o.status='completed' GROUP BY 1)
SELECT SUM(n>=2) repeat_buyers, ROUND(100.0*SUM(n>=2)/COUNT(*),1) pct FROM p;'''))

,customers,buyers,repeat_buyers,repeat_pct_of_all_customers,repeat_pct_of_buyers
0,10,8,4,40.0,50.0


Sensitivity - WITHOUT deduping order 1006 (customer 105 wrongly looks like a repeat buyer):


,repeat_buyers,pct
0,5,50.0


## S5. Month-on-month completed revenue

In [8]:
display(q('''
WITH monthly AS (
  SELECT strftime('%Y-%m', order_date) AS month,
         SUM(amount_kes)               AS revenue_kes
  FROM orders_dedup
  WHERE status='completed'
    AND amount_kes < 50000              -- exclude suspected entry error (1013)
  GROUP BY 1)
SELECT month, revenue_kes,
       LAG(revenue_kes) OVER (ORDER BY month) AS prev_month,
       ROUND(100.0*(revenue_kes - LAG(revenue_kes) OVER (ORDER BY month))
             / LAG(revenue_kes) OVER (ORDER BY month), 1) AS mom_pct
FROM monthly ORDER BY month;'''))
print('Sensitivity - outlier left in:')
display(q('''
WITH monthly AS (SELECT strftime('%Y-%m', order_date) month, SUM(amount_kes) revenue_kes
                 FROM orders_dedup WHERE status='completed' GROUP BY 1)
SELECT month, revenue_kes,
       ROUND(100.0*(revenue_kes-LAG(revenue_kes) OVER (ORDER BY month))/LAG(revenue_kes) OVER (ORDER BY month),1) mom_pct
FROM monthly;'''))

,month,revenue_kes,prev_month,mom_pct
0,2026-06,9000.0,NaN,NaN
1,2026-07,6800.0,9000.0,-24.4
2,2026-08,7000.0,6800.0,2.9
3,2026-09,10300.0,7000.0,47.1


Sensitivity - outlier left in:


,month,revenue_kes,mom_pct
0,2026-06,9000.0,NaN
1,2026-07,6800.0,-24.4
2,2026-08,192000.0,2723.5
3,2026-09,10300.0,-94.6


## D2. Spreadsheet formulas – cross-check in pandas

In [9]:
d = orders.copy()
raw = d[(d.status=='completed') & (d.payment_method=='M-Pesa')].amount_kes.sum()
dd = d.drop_duplicates('order_id')
clean = dd[(dd.status=='completed') & (dd.payment_method=='M-Pesa') & (dd.amount_kes<50000)].amount_kes.sum()
print('SUMIFS on raw sheet     :', raw)
print('After dedupe + outlier  :', clean)

SUMIFS on raw sheet     : 209550.0
After dedupe + outlier  : 23050.0


## D3. Mean vs median of completed order value

In [10]:
v = dd[(dd.status=='completed')].amount_kes.dropna()
clean_v = v[v<50000]
print(f'With outlier   -> mean {v.mean():,.0f} | median {v.median():,.0f}  (n={len(v)})')
print(f'Outlier removed-> mean {clean_v.mean():,.0f} | median {clean_v.median():,.0f}  (n={len(clean_v)})')

With outlier   -> mean 14,540 | median 2,400  (n=15)
Outlier removed-> mean 2,364 | median 2,375  (n=14)


## T2. A/B test: two-proportion z-test

In [11]:
n1,x1,n2,x2 = 1000,50,1000,60
p1,p2 = x1/n1, x2/n2
pp = (x1+x2)/(n1+n2)
se_pool = np.sqrt(pp*(1-pp)*(1/n1+1/n2))
z = (p2-p1)/se_pool
pval = 2*(1-norm.cdf(abs(z)))
se_unp = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
ci = ((p2-p1)-1.96*se_unp, (p2-p1)+1.96*se_unp)
print(f'Conversion: {p1:.1%} -> {p2:.1%} | absolute +{(p2-p1)*100:.1f}pp | relative +{(p2/p1-1):.0%}')
print(f'z = {z:.3f}, two-sided p = {pval:.3f}')
print(f'95% CI for difference: {ci[0]*100:.2f}pp to {ci[1]*100:.2f}pp')

# sample size per arm to detect 5% -> 6% with 80% power, alpha 5%
za, zb = norm.ppf(0.975), norm.ppf(0.80)
n_needed = (za+zb)**2 * (p1*(1-p1)+p2*(1-p2)) / (p2-p1)**2
print(f'Needed per arm for 80% power: ~{n_needed:,.0f}')
# minimum detectable effect with 1,000 per arm
mde = (za+zb)*np.sqrt(2*0.055*0.945/1000)
print(f'Min detectable absolute lift at n=1,000/arm: ~{mde*100:.1f}pp')

Conversion: 5.0% -> 6.0% | absolute +1.0pp | relative +20%
z = 0.981, two-sided p = 0.327
95% CI for difference: -1.00pp to 3.00pp
Needed per arm for 80% power: ~8,155
Min detectable absolute lift at n=1,000/arm: ~2.9pp


## Part 5. Falling conversion – mix shift (Simpson-style) analysis

In [12]:
t = pd.DataFrame({'channel':['Organic','Paid Social'],
                  'sep_sessions':[40000,10000],'sep_orders':[2400,200],
                  'oct_sessions':[38000,30000],'oct_orders':[2318,630]})
t['sep_cvr']=t.sep_orders/t.sep_sessions; t['oct_cvr']=t.oct_orders/t.oct_sessions
t['sep_mix']=t.sep_sessions/t.sep_sessions.sum(); t['oct_mix']=t.oct_sessions/t.oct_sessions.sum()
display(t.round(4))

tot_sep = t.sep_orders.sum()/t.sep_sessions.sum()
tot_oct = t.oct_orders.sum()/t.oct_sessions.sum()
oct_at_sep_mix = (t.sep_mix*t.oct_cvr).sum()
print(f'Blended CVR  Sep {tot_sep:.2%} -> Oct {tot_oct:.2%}  (change {100*(tot_oct-tot_sep):+.2f}pp)')
print(f'Rate effect (within-channel change, Sep mix): {100*(oct_at_sep_mix-tot_sep):+.2f}pp')
print(f'Mix effect (paid share of traffic {t.sep_mix[1]:.0%} -> {t.oct_mix[1]:.0%}): {100*(tot_oct-oct_at_sep_mix):+.2f}pp')
print('Orders change by channel:', (t.oct_orders-t.sep_orders).tolist(), '| total', int(t.oct_orders.sum()-t.sep_orders.sum()))
print(f"Marginal paid conversion on the extra sessions: {(630-200)/(30000-10000):.2%}")

,channel,sep_sessions,sep_orders,oct_sessions,oct_orders,sep_cvr,oct_cvr,sep_mix,oct_mix
0,Organic,40000,2400,38000,2318,0.06,0.061,0.8,0.5588
1,Paid Social,10000,200,30000,630,0.02,0.021,0.2,0.4412


Blended CVR  Sep 5.20% -> Oct 4.34%  (change -0.86pp)
Rate effect (within-channel change, Sep mix): +0.10pp
Mix effect (paid share of traffic 20% -> 44%): -0.96pp
Orders change by channel: [-82, 430] | total 348
Marginal paid conversion on the extra sessions: 2.15%
